In [3]:
import os
import pandas as pd
from qiime2 import Visualization
from qiime2 import Artifact
import matplotlib.pyplot as plt 

%matplotlib inline

In [4]:
data_dir = '/home/jovyan/FunGut/data'

Define input file locations: 

In [5]:
reference_fasta = f"{data_dir}/raw/ITS_database.fasta"
clustered_seqs = f"{data_dir}/processed_ignored/fungut_seqs_closed_ref_ITS.qza"
metadata = f"{data_dir}/raw/fungut_metadata.tsv"
clustered_table = f"{data_dir}/processed_ignored/fungut_table_closed_ref_ITS.qza"

Define the output file location:

In [6]:
taxonomy_output = f"{data_dir}/processed_ignored/fungut_taxonomy_closed_ref_ITS.qza"
taxonomy_visualization = f"{data_dir}/processed/fungut_taxonomy_closed_ref_ITS.qzv"

Inspect the FASTA headers to check for taxonomy labels and how they are separated from reference IDs:

In [5]:
! head -n 4 "$reference_fasta"

>UDB016649|k__Fungi;p__Basidiomycota;c__Agaricomycetes;o__Thelephorales;f__Thelephoraceae;g__Thelephora;s__Thelephora_albomarginata|SH1281904.10FU
TAGAGGAAGTAAAAGTCGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTACCGAATTGTCAAACGGGTTGTTGCTGGTCTTCAAACGGGGACATGTGCACGCTCTGTTTACACATCCACTCACACCTGTGCACCCTCCGTAGTTCTATGGTCTGGGGGACCCTGTCTTCCTGCCGTGGTTCTACGTCTTTACACACACTCTGTAATAAAGTCTTATGGAATGTATGCCGCGTCTAACGCAATACAATACAACTTTCAGCAACGGATCTCTTGGCTCTCGCATCGATGAAGAACGCAGCGAAATGCGATAAGTAATGTGAATTGCAGAATTCAGTGAATCATCGAATCTTTGAACGCACCTTGCGCCCCTTGGCTATTCCGAGGGGCATGCCTGTTTGAGTATCATGAACACCTCAACTCTCATGGTTCGCCATGATGAGCTTGGACTCTGGGGGTTTTGCTGGCCTGTGGTCGGCTCCCCTCAAATGAATCGGCCTCCCAGTGTTTGGTGGCATCACGGGTGTGATAAATATCTACGCTCACGGTTGTCTGCCAGGTAACCTTTGGTGACAAAGGTTCGCTGGAGCTCACAGATGTCTCTCCTCGGCGAGGACAGCTTTTTTGAACGTTCGATCTCAAATCAGGTAGGACTACCCGCTGAACTTAAGCATATCAATA
>UDB016650|k__Fungi;p__Basidiomycota;c__Agaricomycetes;o__Thelephorales;f__Thelephoraceae;g__Tomentella;s__Tomentella_sp|SH1282297.10FU
TAGAGGAAGTAAAAGTCGTAACAAGGTTTCC

Read the reference FASTA into a pandas DataFrame using | as the column separator. Header lines of the FASTA file are divided into reference accession, taxonomy, and UNITE identifier.

In [6]:
reference = pd.read_csv(reference_fasta, sep="|", header=None)

In [7]:
reference.head()

,0,1,2
0,>UDB016649,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1281904.10FU
1,TAGAGGAAGTAAAAGTCGTAACAAGGTTTCCGTAGGTGAACCTGCG...,NaN,NaN
2,>UDB016650,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1282297.10FU
3,TAGAGGAAGTAAAAGTCGTAACAAGGTTTCCGTAGGTGAACCTGCG...,NaN,NaN
4,>UDB016651,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1146664.10FU


Keep only the header rows:

In [8]:
condition = reference[0].str.startswith(">")
headers = reference[condition].copy()
headers.head()

,0,1,2
0,>UDB016649,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1281904.10FU
2,>UDB016650,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1282297.10FU
4,>UDB016651,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1146664.10FU
6,>UDB016653,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1283391.10FU
8,>UDB016654,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1059744.10FU


Extract the taxonomic labels from column 1 into a pandas Series:

In [9]:
taxonomy = headers[1].copy()

Create feature IDs by joining the reference accession and kingdom label with |, without the leading >, so they match the IDs in our clustered table:

In [10]:
feature_ids = headers[0].str.removeprefix(">") + "|" + headers[1].str.split(";").str[0]

Create a table linking each feature ID to its taxonomic label, keeping the labels in a separate column so QIIME 2 can use them:

In [11]:
taxonomy_table = pd.DataFrame({"Feature ID": feature_ids, "Taxon": taxonomy})
taxonomy_table.head()

,Feature ID,Taxon
0,UDB016649|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...
2,UDB016650|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...
4,UDB016651|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...
6,UDB016653|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...
8,UDB016654|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...


Convert the count table from closed-reference clustering into a pandas DataFrame so we can match its feature IDs to the taxonomic labels in our reference database:

In [12]:
table_artifact = Artifact.load(clustered_table)
count_table = table_artifact.view(pd.DataFrame)
count_table.iloc[:3, :3] # visualize first three rows & columns 

,EU932730|k__Fungi,EU871493|k__Fungi,DQ667972|k__Fungi
ERR5327198,137.0,3.0,2.0
ERR5327199,9.0,1172.0,0.0
ERR5327266,40611.0,5.0,1.0


Collect the feature IDs from the clustered count table so we can afterwards filter the reference taxonomy entries relevant to our samples:

In [13]:
clustered_ids = count_table.columns

Select rows from the reference taxonomy table whose feature IDs match those in our clustered feature table.

-> Reference IDs: taxonomy_table["Feature ID"]  
-> Clustered feature IDs: clustered_ids

In [14]:
condition = taxonomy_table["Feature ID"].isin(clustered_ids)
filtered_taxonomy_table = taxonomy_table[condition].copy()
filtered_taxonomy_table.iloc[:5, ]

,Feature ID,Taxon
5240,UDB015028|k__Fungi,k__Fungi;p__Ascomycota;c__Dothideomycetes;o__D...
5526,EF493288|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...
6046,UDB004143|k__Fungi,k__Fungi;p__Ascomycota;c__Leotiomycetes;o__Hel...
7394,GU256755|k__Fungi,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__S...
7716,GU256960|k__Fungi,k__Fungi;p__Basidiomycota;c__Cystobasidiomycet...


Check that every clustered feature ID has a matching entry in the filtered taxonomy table (from the reference database):

In [15]:
missing_ids = clustered_ids.difference(filtered_taxonomy_table["Feature ID"])
print(f"Features without taxonomy: {len(missing_ids)}")

Features without taxonomy: 0


Save the filtered taxonomy table as a TSV file for import into QIIME 2:

In [16]:
taxonomy_tsv = f"{data_dir}/processed_ignored/fungut_taxonomy_closed_ref_ITS.tsv"

filtered_taxonomy_table.to_csv(
    taxonomy_tsv, sep="\t", index=False, header=False
)

Import the saved taxonomy table as a QIIME 2 artifact:

In [17]:
! qiime tools import \
    --type 'FeatureData[Taxonomy]' \
    --input-format HeaderlessTSVTaxonomyFormat \
    --input-path "$taxonomy_tsv" \
    --output-path "$taxonomy_output"

Imported /home/jovyan/FunGut/data/processed_ignored/fungut_taxonomy_closed_ref_ITS.tsv as HeaderlessTSVTaxonomyFormat to /home/jovyan/FunGut/data/processed_ignored/fungut_taxonomy_closed_ref_ITS.qza


Visualize the taxonomy table to analyze which fungal names are linked to our clustered features:

In [18]:
! qiime metadata tabulate \
    --m-input-file "$taxonomy_output" \
    --o-visualization "$taxonomy_visualization"

Saved Visualization to: /home/jovyan/FunGut/data/processed/fungut_taxonomy_closed_ref_ITS.qzv


In [7]:
! qiime taxa barplot \
    --i-table "$clustered_table" \
    --i-taxonomy "$taxonomy_output" \
    --m-metadata-file "$metadata" \
    --o-visualization $data_dir/processed/taxa_barplot_closed_ref_ITS.qzv

Saved Visualization to: /home/jovyan/FunGut/data/processed/taxa_barplot_closed_ref_ITS.qzv


In [9]:
! qiime taxa filter-table \
    --i-table "$clustered_table" \
    --i-taxonomy "$taxonomy_output" \
    --p-exclude mitochondria \
    --p-include c__ \
    --o-filtered-table "$data_dir/processed_ignored/taxonomy_table_filtered.qza"

! qiime taxa filter-seqs \
    --i-sequences "$clustered_seqs" \
    --i-taxonomy "$taxonomy_output" \
    --p-exclude mitochondria \
    --p-include c__ \
    --o-filtered-sequences "$data_dir/processed_ignored/taxonomy_seqs_filtered.qza"

Saved FeatureTable[Frequency] to: /home/jovyan/FunGut/data/processed_ignored/taxonomy_table_filtered.qza
Saved FeatureData[Sequence] to: /home/jovyan/FunGut/data/processed_ignored/taxonomy_seqs_filtered.qza


In [12]:
filtered_taxonomy_table = f"{data_dir}/processed_ignored/taxonomy_table_filtered.qza"
barplot_filtered = f"{data_dir}/processed/taxa_barplot_filtered_closed_ref_ITS.qzv"

! qiime taxa barplot \
    --i-table "$filtered_taxonomy_table" \
    --i-taxonomy "$taxonomy_output" \
    --m-metadata-file "$metadata" \
    --o-visualization "$barplot_filtered"

Saved Visualization to: /home/jovyan/FunGut/data/processed/taxa_barplot_filtered_closed_ref_ITS.qzv
